In [1]:
import sys
from pathlib import Path
from typing import Dict, Any


PROJECT_ROOT = Path("..").resolve()  # notebook is inside notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root added:", PROJECT_ROOT)

from agents.cartographer import graphrag_cartographer
from agents.architect import architect_ensemble
from agents.verifier import execution_verifier
from agents.critic import semantic_verifier
from agents.orchestrator import reflex_orchestrator

print("✅ All agents imported")

✅ Project root added: C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4
🔥 cartographer.py LOADED
✅ All agents imported


In [2]:
from agents.schema_registry import load_schema_for_db

print("before load")

schema = load_schema_for_db(
    db_id="academic",
    dataset="spider"
)

print("after load")
print(schema.keys())


c:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


before load
after load
dict_keys(['graph', 'schema_texts', 'schema_ids', 'faiss_index', 'doc_tokens', 'df', 'schema_text', 'embedder'])


In [5]:
from agents.schema_registry import load_schema_for_db

schema = load_schema_for_db(
    db_id="academic",
    dataset="spider"
)

graph = schema["graph"]
schema_texts = schema["schema_texts"]
schema_ids = schema["schema_ids"]
embedder = schema["embedder"]
faiss_index = schema["faiss_index"]
doc_tokens = schema["doc_tokens"]
df_stats = schema["df"]

print("✅ Schema loaded")
print("Tables / Columns indexed:", len(schema_texts))


✅ Schema loaded
Tables / Columns indexed: 15


In [ ]:
print("HELLO FROM PYTHON")


In [6]:
from pathlib import Path
from llama_cpp import Llama

MODELS_DIR = Path("../models")

llm_large = Llama(
    model_path=str(MODELS_DIR / "deepseek-coder-6.7b-instruct.Q4_K_M.gguf"),
    n_ctx=2048,
    n_threads=8,
    n_batch=256,
)

llm_medium = Llama(
    model_path=str(MODELS_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf"),
    n_ctx=2048,
    n_threads=8,
    n_batch=256,
)

print("✅ Architect LLMs loaded")


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


✅ Architect LLMs loaded


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


In [7]:
class Role:
    VIEWER = "viewer"
    EDITOR = "editor"
    ADMIN = "admin"


ROLE_PERMISSIONS = {
    Role.VIEWER: {"READ"},
    Role.EDITOR: {"READ", "WRITE"},
    Role.ADMIN: {"READ", "WRITE"}
}


def rbac_check(role: str, intent: str) -> bool:
    return intent in ROLE_PERMISSIONS.get(role, set())


In [8]:
def classify_intent_strict(sql: str) -> str:
    sql_l = sql.lower()

    write_keywords = [
        "insert", "update", "delete",
        "create", "drop", "alter", "truncate"
    ]

    if any(k in sql_l for k in write_keywords):
        return "WRITE"

    return "READ"


In [9]:
import re

def simulate_schema_impact(sql: str) -> Dict[str, Any]:
    sql_l = sql.lower()

    tables = set(
        t for pair in re.findall(r"(from|into|update|join)\s+(\w+)", sql_l)
        for t in pair if t not in {"from", "into", "update", "join"}
    )

    destructive = any(
        k in sql_l for k in ["drop", "truncate", "delete"]
    )

    return {
        "tables_affected": list(tables),
        "destructive": destructive
    }


In [10]:
def write_policy_guard(impact: Dict[str, Any]) -> bool:
    # Block destructive ops by default
    if impact["destructive"]:
        return False

    # Empty impact = suspicious
    if not impact["tables_affected"]:
        return False

    return True


In [11]:
from llama_cpp import Llama
from pathlib import Path

CRITIC_MODEL_PATH = Path("../models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf")

critic_llm = Llama(
    model_path=str(CRITIC_MODEL_PATH),
    n_ctx=1024,
    n_threads=8,
    n_batch=128,
    verbose=True   # 🔑 REQUIRED in Jupyter / VS Code
)

print("✅ critic_llm loaded for WRITE safety")


✅ critic_llm loaded for WRITE safety


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


In [12]:
def semantic_verifier_write(
    question: str,
    sql: str,
    llm,
    min_confidence: float = 0.7
):
    prompt = f"""
You are auditing a WRITE database operation.

Decide if this WRITE is:
- Intentional
- Logically aligned with the question
- Non-destructive
- Not a meaningless no-op

Question:
{question}

SQL:
{sql}

Return ONLY JSON:
{{
  "ok": true | false,
  "confidence": number between 0 and 1,
  "reason": "short explanation"
}}
"""
    out = llm(prompt, max_tokens=128)
    text = out["choices"][0]["text"].strip()

    try:
        result = json.loads(text)
    except Exception:
        return {"ok": False, "confidence": 0.0, "reason": "Invalid JSON from WRITE critic"}

    if result["confidence"] < min_confidence:
        result["ok"] = False

    return result


In [13]:
def safe_write_operation(
    *,
    question: str,
    sql: str,
    role: str,
    db_path: Path,
    schema_text: str,
    min_confidence: float = 0.7
) -> Dict[str, Any]:

    verdict = {
        "allowed": False,
        "reason": None
    }

    # 1️⃣ Intent classification
    intent = classify_intent_strict(sql)
    if intent != "WRITE":
        verdict["reason"] = "Not a WRITE operation"
        return verdict
    
    # 1.5️⃣ HARD no-op guard (non-negotiable)
    if is_noop_write(sql):
        verdict["reason"] = "Semantic no-op WRITE blocked"
        return verdict
    
    # 2️⃣ RBAC
    if not rbac_check(role, intent):
        verdict["reason"] = f"RBAC denied for role={role}"
        return verdict

    # 3️⃣ Schema impact simulation
    impact = simulate_schema_impact(sql)
    if not write_policy_guard(impact):
        verdict["reason"] = "Schema impact policy violation"
        return verdict

    # 4️⃣ Dry-run execution (Phase 4)
    exec_v = execution_verifier(
        sql=sql,
        intent="WRITE",
        db_path=db_path
    )

    if not exec_v["allowed"]:
        verdict["reason"] = exec_v["reason"]
        return verdict

    # 5️⃣ Semantic critic approval (Phase 5)
    semantic = semantic_verifier_write(
        question=question,
        sql=sql,
        llm=critic_llm,
        min_confidence=min_confidence
    )

    if not semantic["ok"]:
        verdict["semantic_warning"] = semantic.get("reason", "Low confidence")

    # ✅ ALL AGENTS AGREED
    verdict["allowed"] = True
    verdict["reason"] = "WRITE approved by all agents"
    return verdict


In [14]:
import re

def is_noop_write(sql: str) -> bool:
    sql_l = sql.lower()

    # SET col = col
    if re.search(r"set\s+(\w+)\s*=\s*\1", sql_l):
        return True

    # SET col = TRIM(col) with no WHERE narrowing
    if "trim(" in sql_l and "where" not in sql_l:
        return True

    return False


In [15]:
sql = "DELETE FROM student WHERE age < 18"

result = safe_write_operation(
    question="Remove underage students",
    sql=sql,
    role=Role.VIEWER,
    db_path=Path("../data/spider/database/academic/academic.sqlite"),
    schema_text=""
)

print(result)


{'allowed': False, 'reason': 'RBAC denied for role=viewer'}


In [16]:

sql = """
INSERT INTO author (name)
VALUES ('Temporary Safety Test Author')
"""

result = safe_write_operation(
    question="Add a temporary test author record for validation",
    sql=sql,
    role=Role.EDITOR,
    db_path=Path("../data/spider/database/academic/academic.sqlite"),
    schema_text=""
)

print(result)



{'allowed': True, 'reason': 'WRITE approved by all agents', 'semantic_warning': 'Invalid JSON from WRITE critic'}


In [17]:
from pathlib import Path
import pandas as pd

# -----------------------------
# TEST CASES 
# -----------------------------
TEST_CASES = [
    {
        "name": "RBAC block (viewer)",
        "question": "Remove underage authors",
        "sql": "DELETE FROM author WHERE aid < 10",
        "role": Role.VIEWER,
        "expected_allowed": False,
        "expected_reason_contains": "RBAC"
    },
    {
        "name": "Destructive op blocked",
        "question": "Drop author table",
        "sql": "DROP TABLE author",
        "role": Role.ADMIN,
        "expected_allowed": False,
        "expected_reason_contains": "Schema impact"
    },
    {
        "name": "No-op WRITE blocked",
        "question": "Touch author records safely",
        "sql": "UPDATE author SET name = name WHERE aid >= 0",
        "role": Role.EDITOR,
        "expected_allowed": False,
        "expected_reason_contains": "Semantic"
    },
    {
        "name": "Safe meaningful UPDATE allowed",
        "question": "Normalize author names",
        "sql": "UPDATE author SET name = TRIM(name) WHERE aid >= 0",
        "role": Role.EDITOR,
        "expected_allowed": True,
        "expected_reason_contains": "approved"
    }
]

DB_PATH = Path("../data/spider/database/academic/academic.sqlite")

results = []

for tc in TEST_CASES:
    out = safe_write_operation(
        question=tc["question"],
        sql=tc["sql"],
        role=tc["role"],
        db_path=DB_PATH,
        schema_text=""
    )

    passed = (
        out["allowed"] == tc["expected_allowed"]
        and tc["expected_reason_contains"].lower() in out["reason"].lower()
    )

    results.append({
        "Test Name": tc["name"],
        "Role": tc["role"],
        "SQL": tc["sql"],
        "Allowed": out["allowed"],
        "Reason": out["reason"],
        "Test Passed": passed
    })

# -----------------------------
# FINAL REPORT
# -----------------------------
report_df = pd.DataFrame(results)

print("📊 PHASE 7 — SAFE WRITE OPERATIONS TEST REPORT\n")
display(report_df)

print("\n✅ Overall Status:",
      "PASS" if report_df["Test Passed"].all() else "FAIL")


Llama.generate: prefix-match hit


📊 PHASE 7 — SAFE WRITE OPERATIONS TEST REPORT



,Test Name,Role,SQL,Allowed,Reason,Test Passed
0,RBAC block (viewer),viewer,DELETE FROM author WHERE aid < 10,False,RBAC denied for role=viewer,True
1,Destructive op blocked,admin,DROP TABLE author,False,Schema impact policy violation,True
2,No-op WRITE blocked,editor,UPDATE author SET name = name WHERE aid >= 0,False,Semantic no-op WRITE blocked,True
3,Safe meaningful UPDATE allowed,editor,UPDATE author SET name = TRIM(name) WHERE aid ...,True,WRITE approved by all agents,True



✅ Overall Status: PASS
